In [4]:
!pip install mysql-connector-python

  Using cached mysql_connector_python-9.6.0-cp312-cp312-win_amd64.whl.metadata (11 kB)
Using cached mysql_connector_python-9.6.0-cp312-cp312-win_amd64.whl (16.5 MB)


In [5]:
import pandas as pd

school_df = pd.read_csv('school_df.csv')
requirement_df = pd.read_csv('requirement_df.csv')
admission_df = pd.read_csv('admission_df.csv')
faq_df = pd.read_csv('faq_df.csv')

print(school_df.columns)
print(admission_df.columns)
print(requirement_df.columns)
print(faq_df.columns)

Index(['name', 'country', 'location'], dtype='str')
Index(['name', 'regular_deadline', 'early_deadline', 'tuition'], dtype='str')
Index(['name', 'type', 'metric_type', 'require', 'value', 'notes'], dtype='str')
Index(['name', 'question', 'answer', 'category'], dtype='str')


In [6]:
import mysql.connector


# -------------SQL 연결-----------------#
connection = mysql.connector.connect(
    host = 'localhost', # MySQL 서버 주소(iP)
    user = 'ohgiraffers', # 사용자 이름
    password = 'ohgiraffers', # 비밀번호
    database = 'universitydb' #사용할 DB 스키마
)
cursor = connection.cursor(buffered=True)
# -------------------------------------#
cursor.execute("SELECT DATABASE();")
print(cursor.fetchone())

('universitydb',)


In [7]:
# 2. 이제 다시 실행 (주의: cursor = cursor.execute() 라고 쓰면 안 됩니다!)
cursor.execute('SHOW TABLES')

# 3. 결과 출력
for table in cursor:
    print(table)
school_df_sql = """INSERT INTO school_info(school_name, country , location)  
                    VALUE (%s, %s, %s)"""
# for _, row in school_df.iterrows(): # 데이터프레임을 한줄씩 쓰는것
#     cursor.execute(school_df_sql, (
#         row['name'],
#         row['country'],
#         row['location']))
    
school_data = school_df[['name', 'country', 'location']].values.tolist()
cursor.executemany(school_df_sql, school_data)
connection.commit()

print("school 완료")




('admission_info',)
('faq_info',)
('requirement_info',)
('school_info',)
school 완료


In [8]:
cursor.execute('SELECT * FROM school_info')

rows = cursor.fetchall()

for row in rows:
    print(row)


(1, 'New York University', 'USA', 'New York, NY')
(2, 'University of Southern California', 'USA', 'Los Angeles, California')
(3, 'University of Illinois Urbana-Champaign', 'USA', 'Urbana, IL 61801')
(4, 'Columbia University', 'USA', 'New York, NY')
(5, 'University of California, Los Angeles', 'USA', 'Los Angeles, CA')
(6, 'Boston University', 'USA', 'Boston, MA')
(7, 'University of California, Berkeley', 'USA', 'Berkeley, CA')
(8, 'University of California, San Diego', 'USA', 'La Jolla, CA')
(9, 'Purdue University', 'USA', 'West Lafayette, IN')
(10, 'The Pennsylvania State University', 'USA', 'University Park, PA')


In [9]:

cursor.execute('SELECT * FROM school_info')
rows = cursor.fetchall()
school_id_map = {row[1]: row[0] for row in rows}
admission_df['school_id'] = admission_df['name'].map(school_id_map) # 이름으로 primary key 추가

In [10]:


admission_df_sql = """INSERT INTO admission_info(school_id, tuition , regular_deadline_date, early_deadline_date)  
                    VALUE (%s, %s, %s, %s)"""

cols = ['school_id', 'tuition', 'regular_deadline', 'early_deadline']

adm_data = admission_df[cols] \
    .astype(object) \
    .where(pd.notnull(admission_df[cols]), None) \
    .values.tolist()

# 6️⃣ INSERT
cursor.executemany(admission_df_sql, adm_data)
connection.commit()

print("완료")

완료


In [11]:
cursor.execute('SELECT * FROM admission_info')

rows = cursor.fetchall()

for row in rows:
    print(row)

(1, 1, 60000, datetime.date(2026, 1, 1), datetime.date(2026, 11, 1))
(2, 2, 75384, datetime.date(2026, 1, 10), datetime.date(2026, 11, 1))
(3, 3, 58316, datetime.date(2026, 1, 5), datetime.date(2026, 11, 1))
(4, 4, 70170, datetime.date(2026, 1, 1), datetime.date(2026, 11, 1))
(5, 5, 80739, datetime.date(2026, 12, 1), None)
(6, 6, 73024, datetime.date(2026, 1, 5), datetime.date(2026, 11, 2))
(7, 7, 79881, datetime.date(2026, 11, 30), None)
(8, 8, 21546, datetime.date(2026, 12, 1), None)
(9, 9, 30000, datetime.date(2026, 1, 1), datetime.date(2026, 11, 1))
(10, 10, 92000, datetime.date(2026, 12, 1), datetime.date(2026, 11, 1))


In [12]:

requirement_df['school_id'] = requirement_df['name'].map(school_id_map) # 이름으로 primary key 추가

total_requirement_df_sql = """INSERT INTO requirement_info(school_id, requirement_type , metric_type, requirement_require, requirement_value, notes)  
                    VALUE (%s, %s, %s, %s, %s, %s)"""

cols = ['school_id', 'type', 'metric_type', 'require', 'value', 'notes']

req_data = requirement_df[cols] \
    .astype(object) \
    .where(pd.notnull(requirement_df[cols]), None) \
    .values.tolist()

cursor.executemany(total_requirement_df_sql, req_data)
connection.commit()

print("완료")

완료


In [13]:
cursor.execute('SELECT * FROM requirement_info')

rows = cursor.fetchall()

for row in rows:
    print(row)

(1, 1, 'TOEFL', 'Competitive Score', 'Conditionally Required', '100.0', 'No strict minimum, but this is the competitive score.')
(2, 1, 'IELTS', 'Competitive Score', 'Conditionally Required', '7.5', 'No strict minimum, but this is the competitive score.')
(3, 1, 'Duolingo', 'Competitive Score', 'Conditionally Required', '135.0', 'No strict minimum, but this is the competitive score.')
(4, 1, 'PTE', 'Competitive Score', 'Conditionally Required', '70.0', 'No strict minimum, but this is the competitive score.')
(5, 1, 'Cambridge English', 'Competitive Score', 'Conditionally Required', '191.0', 'No strict minimum, but this is the competitive score.')
(6, 1, 'iTEP', 'Competitive Score', 'Conditionally Required', '4.5', 'No strict minimum, but this is the competitive score.')
(7, 2, 'Common Application', None, 'Required', None, 'Completed application form is required.')
(8, 2, 'Official Test Scores', None, 'Optional', None, 'For applicants who choose to submit test scores, they will be consi

In [14]:
cursor.execute('SELECT * FROM school_info')
rows = cursor.fetchall()
school_id_map = {row[1]: row[0] for row in rows}
faq_df['school_id'] = faq_df['name'].map(school_id_map) # 이름으로 primary key 추가
faq_df


,name,question,answer,category,school_id
0,New York University,What are the key deadlines for undergraduate a...,"For undergraduate admissions at NYU, the key d...",Admissions,1
1,New York University,Is NYU test-optional?,"Yes, NYU is test-optional through the 2026-202...",Admissions,1
2,New York University,Does NYU meet 100% of demonstrated financial n...,"Yes, NYU will meet 100% of demonstrated financ...",Financial Aid,1
3,New York University,What English proficiency exams does NYU accept...,Non-native speakers may need to submit scores ...,International Applicants,1
4,New York University,What is the benefit of Spring Admission at NYU?,Spring Admission allows students to start thei...,Spring Admissions,1
5,University of Southern California,What are the important application deadlines f...,USC has the following key deadlines: Early Act...,Dates and Deadlines,2
6,University of Southern California,What is included in the application checklist ...,The application checklist for USC includes: Th...,Admission Requirements,2
7,University of Southern California,Does USC accept home-schooled students differe...,"Yes, home-schooled students applying to USC mu...",Application Process,2
8,University of Southern California,Does USC offer need-based financial aid to int...,"No, USC does not provide need-based financial ...",Financial Aid,2
9,University of Southern California,What are the English proficiency requirements ...,International applicants whose native language...,International Students,2


In [15]:
faq_df_sql = """INSERT INTO faq_info(school_id, question , answer, category)  
                    VALUE (%s, %s, %s, %s)"""

cols = ['school_id', 'question', 'answer', 'category']

faq_data = faq_df[cols] \
    .astype(object) \
    .where(pd.notnull(faq_df[cols]), None) \
    .values.tolist()

# 6️⃣ INSERT
cursor.executemany(faq_df_sql, faq_data)
connection.commit()

print("완료")

완료


In [16]:
cursor.execute('SELECT * FROM faq_info')

rows = cursor.fetchall()

for row in rows:
    print(row)

(1, 1, 'What are the key deadlines for undergraduate admissions?', 'For undergraduate admissions at NYU, the key deadlines are as follows: Early Decision I by November 1 with decisions released by December 15, Early Decision II by January 1 with decisions released by February 15, and Regular Decision by January 5 with decisions released by April 1.', 'Admissions')
(2, 1, 'Is NYU test-optional?', 'Yes, NYU is test-optional through the 2026-2027 application cycle. Students can choose whether to submit standardized testing as part of their application and may only need to submit one test.', 'Admissions')
(3, 1, 'Does NYU meet 100% of demonstrated financial need?', 'Yes, NYU will meet 100% of demonstrated financial need for all first-time, first-year undergraduate students admitted to the New York campus.', 'Financial Aid')
(4, 1, 'What English proficiency exams does NYU accept for non-native speakers?', 'Non-native speakers may need to submit scores from recognized English proficiency exa